In [ ]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import joblib
from src.data_utils import load_config, load_raw_data, clean_data, save_processed
from src.features import encode_categoricals, scale_features, engineer_features, get_feature_matrix

config = load_config("../config/config.yaml")
df_raw = load_raw_data(config)

In [ ]:

df_clean = clean_data(df_raw, config)
print(f"\nShape final: {df_clean.shape}")

In [ ]:

df_feat = engineer_features(df_clean)
print("Features creadas:")
print([c for c in df_feat.columns if c not in df_raw.columns])

In [ ]:
df_feat, encoders = encode_categoricals(df_feat, config["features_categoricas"])
joblib.dump(encoders, "../models/label_encoders.pkl")
print(" Encoders guardados")

In [ ]:

X = get_feature_matrix(df_feat, config, use_log=True)
y_reg  = np.log1p(df_feat[config["target_regression"]])   # log-transform para regresión
y_clf  = df_feat[config["target_classification"]]

X_train, X_test, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=config["test_size"], random_state=config["random_state"])
_, _, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=config["test_size"], random_state=config["random_state"])

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance viral en train: {y_train_c.mean():.2%}")

In [ ]:

X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

In [ ]:

import os; os.makedirs("../data/processed", exist_ok=True)
joblib.dump((X_train, X_test, X_train_sc, X_test_sc,
             y_train_r, y_test_r, y_train_c, y_test_c,
             list(X.columns)), "../data/processed/splits.pkl")

save_processed(df_feat, config)
print("\nPreparación completada. Datos listos para modelado.")